# Employee Promotion Prediction Using Ensemble Learning
### Comparative Study of Bagging, Boosting, Voting and Stacking

**Student:** Abhishek Phatangare  
**Course:** M.Sc. Computer Science  
**Target:** `promoted`

This is a new version of the Employee Promotion Prediction project. The same employee dataset is studied from an **Ensemble Learning** perspective. The project compares baseline models, Bagging, Boosting, Voting and Stacking.

## 1. Problem Statement

Build a Machine Learning classification system that predicts whether an employee is likely to be promoted using employee performance, experience, training, skills, productivity, engagement and other attributes.

### Research Question
**Can ensemble learning techniques improve employee promotion prediction compared with individual Machine Learning models?**

## 2. Ensemble Methods Covered

- **Baseline:** Logistic Regression, Decision Tree
- **Bagging:** Random Forest, Extra Trees
- **Boosting:** AdaBoost, Gradient Boosting, XGBoost, HistGradientBoosting
- **Voting:** Hard Voting, Soft Voting
- **Stacking:** Random Forest + Extra Trees + XGBoost with Logistic Regression as meta learner
- **Evaluation:** Accuracy, Precision, Recall, F1, ROC-AUC, confusion matrix and 5-fold cross-validation

## 3. Import Libraries

In [ ]:
# If needed: pip install pandas numpy matplotlib seaborn scikit-learn xgboost joblib
import os, time, warnings, joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier, VotingClassifier, StackingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report, roc_curve
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
RANDOM_STATE=42
TARGET='promoted'
print('Libraries imported successfully')

## 4. Load Dataset

In [ ]:
DATA_PATH='data/raw/employee_promotion.csv'
paths=[DATA_PATH,'employee_promotion.csv','Employee_Promotion.csv','employee_data.csv','/mnt/data/employee_promotion.csv']
found=[p for p in paths if os.path.exists(p)]
if not found: raise FileNotFoundError("Dataset not found. Put the CSV at data/raw/employee_promotion.csv or change DATA_PATH.")
DATA_PATH=found[0]
df=pd.read_csv(DATA_PATH)
print('Loaded:',DATA_PATH,' Shape:',df.shape)
display(df.head())

## 5. Dataset Structure, Missing Values and Duplicates

In [ ]:
print('Shape:',df.shape)
print('Columns:',df.columns.tolist())
display(df.dtypes.to_frame('dtype'))
display(df.describe(include='all').T)
missing=pd.DataFrame({'count':df.isna().sum(),'percent':(df.isna().mean()*100).round(2)}).sort_values('count',ascending=False)
display(missing[missing['count']>0])
print('Duplicate rows:',df.duplicated().sum())

## 6. Data Cleaning

In [ ]:
df=df.drop_duplicates().copy()
if TARGET not in df.columns: raise ValueError(f"Target '{TARGET}' not found: {df.columns.tolist()}")
if df[TARGET].dtype=='object':
    mp={'yes':1,'no':0,'promoted':1,'not promoted':0,'true':1,'false':0}
    df[TARGET]=df[TARGET].astype(str).str.strip().str.lower().map(mp)
df=df.dropna(subset=[TARGET]).copy()
df[TARGET]=df[TARGET].astype(int)
print(df.shape)
print(df[TARGET].value_counts())

## 7. Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(7,5)); sns.countplot(data=df,x=TARGET); plt.title('Promotion Distribution'); plt.xticks([0,1],['Not Promoted','Promoted']); plt.show()
for col in ['department','education_level','employment_type']:
    if col in df.columns:
        rate=df.groupby(col)[TARGET].mean().sort_values(ascending=False)
        plt.figure(figsize=(9,5)); sns.barplot(x=rate.index,y=rate.values); plt.title(f'Promotion Rate by {col}'); plt.xticks(rotation=45); plt.ylabel('Promotion Rate'); plt.show()
for col in ['performance_score','manager_rating','kpi_achievement_percent','training_hours_last_year','years_at_company','years_since_last_promotion','skill_assessment_score','employee_engagement_score','job_satisfaction_score','attendance_rate']:
    if col in df.columns:
        plt.figure(figsize=(7,5)); sns.boxplot(data=df,x=TARGET,y=col); plt.title(f'{col} vs Promotion'); plt.show()
num=df.select_dtypes(include=np.number)
plt.figure(figsize=(15,10)); sns.heatmap(num.corr(),center=0,cmap='coolwarm'); plt.title('Correlation Heatmap'); plt.show()

## 8. Prepare Features and Target

In [ ]:
X=df.drop(columns=[TARGET]).copy(); y=df[TARGET].copy()
if 'employee_id' in X.columns: X=X.drop(columns=['employee_id'])
cat=X.select_dtypes(include=['object','category','bool']).columns.tolist()
num=X.select_dtypes(include=np.number).columns.tolist()
print('Categorical:',cat); print('Numerical:',num)
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,stratify=y,random_state=RANDOM_STATE)
pre=ColumnTransformer([('num',Pipeline([('imputer',SimpleImputer(strategy='median')),('scale',StandardScaler())]),num),('cat',Pipeline([('imputer',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])
print(X_train.shape,X_test.shape)

## 9. Baseline Models

In [ ]:
lr=Pipeline([('pre',pre),('model',LogisticRegression(max_iter=2000,class_weight='balanced',random_state=RANDOM_STATE))])
dt=Pipeline([('pre',pre),('model',DecisionTreeClassifier(max_depth=8,class_weight='balanced',random_state=RANDOM_STATE))])

## 10. Bagging Models

In [ ]:
rf=Pipeline([('pre',pre),('model',RandomForestClassifier(n_estimators=300,class_weight='balanced',random_state=RANDOM_STATE,n_jobs=-1))])
extra=Pipeline([('pre',pre),('model',ExtraTreesClassifier(n_estimators=300,class_weight='balanced',random_state=RANDOM_STATE,n_jobs=-1))])

## 11. Boosting Models

In [ ]:
ada=Pipeline([('pre',pre),('model',AdaBoostClassifier(n_estimators=200,learning_rate=.05,random_state=RANDOM_STATE))])
gb=Pipeline([('pre',pre),('model',GradientBoostingClassifier(n_estimators=200,learning_rate=.05,max_depth=3,random_state=RANDOM_STATE))])
xgb=Pipeline([('pre',pre),('model',XGBClassifier(n_estimators=300,max_depth=5,learning_rate=.05,subsample=.8,colsample_bytree=.8,eval_metric='logloss',random_state=RANDOM_STATE,n_jobs=-1))])
hgb=Pipeline([('pre',pre),('model',HistGradientBoostingClassifier(max_iter=200,learning_rate=.05,max_leaf_nodes=31,random_state=RANDOM_STATE))])

## 12. Voting Ensembles

In [ ]:
hard_vote=VotingClassifier([('lr',lr),('rf',rf),('xgb',xgb)],voting='hard',n_jobs=-1)
soft_vote=VotingClassifier([('lr',lr),('rf',rf),('xgb',xgb)],voting='soft',weights=[1,2,2],n_jobs=-1)

## 13. Stacking Ensemble

In [ ]:
stack=StackingClassifier([('rf',rf),('extra',extra),('xgb',xgb)],final_estimator=LogisticRegression(max_iter=2000,class_weight='balanced',random_state=RANDOM_STATE),stack_method='predict_proba',cv=5,n_jobs=-1)

## 14. Train All Models and Compare

In [ ]:
models={'Logistic Regression':lr,'Decision Tree':dt,'Random Forest (Bagging)':rf,'Extra Trees (Bagging)':extra,'AdaBoost':ada,'Gradient Boosting':gb,'XGBoost':xgb,'HistGradientBoosting':hgb,'Hard Voting':hard_vote,'Soft Voting':soft_vote,'Stacking':stack}
results=[]; trained={}
for name,model in models.items():
    t=time.perf_counter(); model.fit(X_train,y_train); elapsed=time.perf_counter()-t
    pred=model.predict(X_test); proba=model.predict_proba(X_test)[:,1] if hasattr(model,'predict_proba') else None
    results.append({'Model':name,'Accuracy':accuracy_score(y_test,pred),'Precision':precision_score(y_test,pred,zero_division=0),'Recall':recall_score(y_test,pred,zero_division=0),'F1-Score':f1_score(y_test,pred,zero_division=0),'ROC-AUC':roc_auc_score(y_test,proba) if proba is not None else np.nan,'Training Time (sec)':elapsed})
    trained[name]=model
results_df=pd.DataFrame(results).sort_values(['F1-Score','ROC-AUC'],ascending=False).reset_index(drop=True)
display(results_df.style.format({c:'{:.4f}' for c in ['Accuracy','Precision','Recall','F1-Score','ROC-AUC']}))

## 15. Model Comparison Plot

In [ ]:
p=results_df.sort_values('F1-Score')
plt.figure(figsize=(11,7)); sns.barplot(data=p,x='F1-Score',y='Model'); plt.xlim(0,1); plt.title('Ensemble Model Comparison - F1 Score'); plt.show()

## 16. Confusion Matrices

In [ ]:
important=['Random Forest (Bagging)','Extra Trees (Bagging)','AdaBoost','Gradient Boosting','XGBoost','Hard Voting','Soft Voting','Stacking']
for name in important:
    cm=confusion_matrix(y_test,trained[name].predict(X_test))
    plt.figure(figsize=(5,4)); sns.heatmap(cm,annot=True,fmt='d',cmap='Blues',xticklabels=['Not Promoted','Promoted'],yticklabels=['Not Promoted','Promoted']); plt.title(name); plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.show()

## 17. Classification Report for Best Model

In [ ]:
best_name=results_df.iloc[0]['Model']; final_model=trained[best_name]
print('Best model:',best_name)
print(classification_report(y_test,final_model.predict(X_test),target_names=['Not Promoted','Promoted'],zero_division=0))

## 18. ROC Curve Comparison

In [ ]:
plt.figure(figsize=(10,7))
for name in ['Random Forest (Bagging)','Extra Trees (Bagging)','XGBoost','Soft Voting','Stacking']:
    proba=trained[name].predict_proba(X_test)[:,1]; fpr,tpr,_=roc_curve(y_test,proba); auc=roc_auc_score(y_test,proba); plt.plot(fpr,tpr,label=f'{name} (AUC={auc:.3f})')
plt.plot([0,1],[0,1],'--'); plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC Curve Comparison'); plt.legend(); plt.show()

## 19. 5-Fold Cross-Validation

In [ ]:
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=RANDOM_STATE)
cv_models={k:models[k] for k in ['Random Forest (Bagging)','Extra Trees (Bagging)','AdaBoost','Gradient Boosting','XGBoost','Hard Voting','Soft Voting','Stacking']}
rows=[]
for name,model in cv_models.items():
    s=cross_validate(model,X,y,cv=cv,scoring=['accuracy','precision','recall','f1','roc_auc'],n_jobs=-1)
    rows.append({'Model':name,'CV Accuracy':s['test_accuracy'].mean(),'CV Precision':s['test_precision'].mean(),'CV Recall':s['test_recall'].mean(),'CV F1':s['test_f1'].mean(),'CV ROC-AUC':s['test_roc_auc'].mean()})
cv_df=pd.DataFrame(rows).sort_values(['CV F1','CV ROC-AUC'],ascending=False).reset_index(drop=True)
display(cv_df.style.format({c:'{:.4f}' for c in cv_df.columns if c!='Model'}))

## 20. Feature Importance

In [ ]:
rf_imp=Pipeline([('pre',pre),('model',RandomForestClassifier(n_estimators=300,class_weight='balanced',random_state=RANDOM_STATE,n_jobs=-1))])
rf_imp.fit(X_train,y_train)
names=rf_imp.named_steps['pre'].get_feature_names_out(); vals=rf_imp.named_steps['model'].feature_importances_
imp=pd.DataFrame({'Feature':names,'Importance':vals}).sort_values('Importance',ascending=False)
display(imp.head(20))
plt.figure(figsize=(10,8)); sns.barplot(data=imp.head(15),x='Importance',y='Feature'); plt.title('Top 15 Feature Importances - Random Forest'); plt.show()

## 21. Final Model Selection and Save

In [ ]:
best_name=results_df.iloc[0]['Model']; final_model=trained[best_name]
os.makedirs('models',exist_ok=True); os.makedirs('outputs',exist_ok=True)
joblib.dump(final_model,'models/employee_promotion_ensemble_model.pkl')
results_df.to_csv('outputs/ensemble_model_comparison.csv',index=False); cv_df.to_csv('outputs/ensemble_cross_validation.csv',index=False)
print('Final model:',best_name); print('Saved model and comparison files.')

## 22. New Employee Prediction

In [ ]:
# Creates a valid example using the dataset's median/mode values. Replace values with a real test case when presenting.
new={c:(X[c].median() if c in num else X[c].mode(dropna=True).iloc[0]) for c in X.columns}
new_df=pd.DataFrame([new])
pred=final_model.predict(new_df)[0]; prob=final_model.predict_proba(new_df)[0,1]
display(new_df)
print('Prediction:', 'Promoted' if pred==1 else 'Not Promoted')
print('Promotion probability:',f'{prob:.2%}')

## 23. Conclusion

The notebook compares individual learners with Bagging, Boosting, Voting and Stacking. The final model is selected using actual test and cross-validation results. **Do not enter fabricated metrics in the report.** The model is an educational/analytical prototype and should not be the sole basis for real employee promotion decisions.

## 24. Expected Project Structure

```text
Employee-Promotion-Ensemble-Learning/
├── data/raw/employee_promotion.csv
├── notebooks/Employee_Promotion_Ensemble_Learning.ipynb
├── models/employee_promotion_ensemble_model.pkl
├── outputs/ensemble_model_comparison.csv
├── outputs/ensemble_cross_validation.csv
├── app.py
├── requirements.txt
└── README.md
```